### Import Packages

In [1]:
import io
import os
import requests
import pathlib
import gzip

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from skimage.io import imsave, imread
import cv2

from hpacellseg.utils import label_cell, label_nuclei

### Cell Segmentation of a Image 
* HPA Segmentation Tool: https://github.com/CellProfiling/HPA-Cell-Segmentation/tree/master

In [2]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"running on device: {DEVICE}")

running on device: cuda


In [3]:
import collections
collections.Iterable = collections.abc.Iterable

import numpy as np
np.bool = np.bool_

import torch
import torch.nn.functional as F

import cv2
from skimage import transform as sk_transform, util as sk_util

import hpacellseg.cellsegmentator as cellsegmentator

NORMALIZE = {"mean": [124 / 255, 117 / 255, 104 / 255], "std": [1 / (0.0167 * 255)] * 3}

NUC_MODEL = "./nuclei-model.pth"
CELL_MODEL = "./cell-model.pth"


class CellSegmentator(cellsegmentator.CellSegmentator):
    def pred_nuclei(self, images):
        # This first is the version from hpacellseg
        # def _preprocess(image):
        #     if isinstance(image, str):
        #         image = imageio.imread(image)
        #     self.target_shape = image.shape
        #     if len(image.shape) == 2:
        #         image = np.dstack((image, image, image))
        #     image = transform.rescale(image, self.scale_factor, multichannel=True)
        #     nuc_image = np.dstack((image[..., 2], image[..., 2], image[..., 2]))
        #     if self.padding:
        #         rows, cols = nuc_image.shape[:2]
        #         self.scaled_shape = rows, cols
        #         nuc_image = cv2.copyMakeBorder(
        #             nuc_image,
        #             32,
        #             (32 - rows % 32),
        #             32,
        #             (32 - cols % 32),
        #             cv2.BORDER_REFLECT,
        #         )
        #     nuc_image = nuc_image.transpose([2, 0, 1])
        #     return nuc_image

        def _preprocess(image):
            if isinstance(image, str):
                image = imageio.imread(image)
            self.target_shape = image.shape
            if len(image.shape) == 2:
                image = np.dstack((image, image, image))
            image = sk_transform.rescale(image, self.scale_factor, channel_axis=-1)
            nuc_image = np.dstack((image[..., 2], image[..., 2], image[..., 2]))
            if self.padding:
                rows, cols = nuc_image.shape[:2]
                self.scaled_shape = rows, cols
                nuc_image = cv2.copyMakeBorder(
                    nuc_image,
                    32,
                    (32 - rows % 32),
                    32,
                    (32 - cols % 32),
                    cv2.BORDER_REFLECT,
                )
            nuc_image = nuc_image.transpose([2, 0, 1])
            return nuc_image
        
        # This first is the version from hpacellseg
        # def _segment_helper(imgs):
        #     with torch.no_grad():
        #         mean = torch.as_tensor(NORMALIZE["mean"], device=self.device)
        #         std = torch.as_tensor(NORMALIZE["std"], device=self.device)
        #         imgs = torch.tensor(np.array(imgs)).float()
        #         imgs = imgs.to(self.device)
        #         imgs = imgs.sub_(mean[:, None, None]).div_(std[:, None, None])
        #         imgs = self.nuclei_model(imgs)
        #         imgs = F.softmax(imgs, dim=1)
        #         return imgs

        # preprocessed_imgs = map(_preprocess, images)
        # predictions = map(lambda x: _segment_helper([x]), preprocessed_imgs)
        # predictions = map(lambda x: x.to("cpu").numpy()[0], predictions)
        # predictions = map(util.img_as_ubyte, predictions)
        # predictions = list(map(self._restore_scaling_padding, predictions))
        # return predictions

        def _segment_helper(imgs):
            with torch.no_grad():
                mean = torch.as_tensor(NORMALIZE["mean"], device=self.device)
                std = torch.as_tensor(NORMALIZE["std"], device=self.device)
                imgs = torch.tensor(np.array(imgs)).float()
                imgs = imgs.to(self.device)
                imgs = imgs.sub_(mean[:, None, None]).div_(std[:, None, None])
                imgs = self.nuclei_model(imgs)
                imgs = F.softmax(imgs, dim=1)
                return imgs

        preprocessed_imgs = map(_preprocess, images)
        predictions = map(lambda x: _segment_helper([x]), preprocessed_imgs)
        predictions = map(lambda x: x.to("cpu").numpy()[0], predictions)
        predictions = map(sk_util.img_as_ubyte, predictions)
        predictions = list(map(self._restore_scaling_padding, predictions))
        return predictions

    def pred_cells(self, images, precombined=False):
        # This first is the version from hpacellseg
        # def _preprocess(image):
        #     self.target_shape = image.shape
        #     if not len(image.shape) == 3:
        #         raise ValueError("image should has 3 channels")
        #     cell_image = transform.rescale(image, self.scale_factor, multichannel=True)
        #     if self.padding:
        #         rows, cols = cell_image.shape[:2]
        #         self.scaled_shape = rows, cols
        #         cell_image = cv2.copyMakeBorder(
        #             cell_image,
        #             32,
        #             (32 - rows % 32),
        #             32,
        #             (32 - cols % 32),
        #             cv2.BORDER_REFLECT,
        #         )
        #     cell_image = cell_image.transpose([2, 0, 1])
        #     return cell_image

        # def _segment_helper(imgs):
        #     with torch.no_grad():
        #         mean = torch.as_tensor(NORMALIZE["mean"], device=self.device)
        #         std = torch.as_tensor(NORMALIZE["std"], device=self.device)
        #         imgs = torch.tensor(np.array(imgs)).float()
        #         imgs = imgs.to(self.device)
        #         imgs = imgs.sub_(mean[:, None, None]).div_(std[:, None, None])

        #         imgs = self.cell_model(imgs)
        #         imgs = F.softmax(imgs, dim=1)
        #         return imgs

        # if not precombined:
        #     images = self._image_conversion(images)
        # preprocessed_imgs = map(_preprocess, images)
        # predictions = map(lambda x: _segment_helper([x]), preprocessed_imgs)
        # predictions = map(lambda x: x.to("cpu").numpy()[0], predictions)
        # predictions = map(self._restore_scaling_padding, predictions)
        # predictions = list(map(util.img_as_ubyte, predictions))

        def _preprocess(image):
            self.target_shape = image.shape
            if not len(image.shape) == 3:
                raise ValueError("image should has 3 channels")
            cell_image = sk_transform.rescale(image, self.scale_factor, channel_axis=-1)
            if self.padding:
                rows, cols = cell_image.shape[:2]
                self.scaled_shape = rows, cols
                cell_image = cv2.copyMakeBorder(
                    cell_image,
                    32,
                    (32 - rows % 32),
                    32,
                    (32 - cols % 32),
                    cv2.BORDER_REFLECT,
                )
            cell_image = cell_image.transpose([2, 0, 1])
            return cell_image

        def _segment_helper(imgs):
            with torch.no_grad():
                mean = torch.as_tensor(NORMALIZE["mean"], device=self.device)
                std = torch.as_tensor(NORMALIZE["std"], device=self.device)
                imgs = torch.tensor(np.array(imgs)).float()
                imgs = imgs.to(self.device)
                imgs = imgs.sub_(mean[:, None, None]).div_(std[:, None, None])

                imgs = self.cell_model(imgs)
                imgs = F.softmax(imgs, dim=1)
                return imgs

        if not precombined:
            images = self._image_conversion(images)
        preprocessed_imgs = map(_preprocess, images)
        predictions = map(lambda x: _segment_helper([x]), preprocessed_imgs)
        predictions = map(lambda x: x.to("cpu").numpy()[0], predictions)
        predictions = map(self._restore_scaling_padding, predictions)
        predictions = list(map(sk_util.img_as_ubyte, predictions))

        return predictions

In [4]:
segmentator = CellSegmentator(
    NUC_MODEL,
    CELL_MODEL,
    scale_factor=0.25,
    device=DEVICE,
    padding=True,
    multi_channel_model=True,
)

please compile abn


/home/jumidlej/git-projects/.venv/lib/python3.12/site-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'pytorch_zoo.unet.DPNUnet' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
/home/jumidlej/git-projects/.venv/lib/python3.12/site-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'torch.nn.modules.container.ModuleList' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
/home/jumidlej/git-projects/.venv/lib/python3.12/site-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve

#### Example of Usage

In [5]:
# # Example of usage
# import hpacellseg.cellsegmentator as cellsegmentator
# from hpacellseg.utils import label_cell, label_nuclei
# from imageio import imwrite

# # Assuming that there are images in the current folder with the
# # following names.
# images = [
#     ["microtubules_one.tif", "microtubules_two.tif"],
#     ["endoplasmic_reticulum_one.tif", "endoplasmic_reticulum_two.tif"],
#     ["nuclei_one.tif", "nuclei_two.tif"]
# ]
# NUC_MODEL = "./nuclei-model.pth"
# CELL_MODEL = "./cell-model.pth"
# # segmentator = cellsegmentator.CellSegmentator(
# #     NUC_MODEL,
# #     CELL_MODEL,
# #     scale_factor=0.25,
# #     device="cuda",
# #     # NOTE: setting padding=True seems to solve most issues that have been encountered
# #     #       during our single cell Kaggle challenge.
# #     padding=False,
# #     multi_channel_model=True,
# # )

# # For nuclei: taking in nuclei channels as inputs
# nuc_segmentations = segmentator.pred_nuclei(images[2])

# # For full cells: taking in 3 channels as inputs
# cell_segmentations = segmentator.pred_cells(images)

# # post-processing nuclei mask
# nuclei_mask = label_nuclei(nuc_segmentations[0])

# # post-processing nuclei and cell mask
# for i, (nuc_segmentation, cell_segmentation) in enumerate(zip(nuc_segmentations, cell_segmentations)):
#     nuclei_mask, cell_mask = label_cell(nuc_segmentation, cell_segmentation)
#     # Save these masks in local folder
#     imwrite(f"nucleimask_{i}.png", nuclei_mask)
#     imwrite(f"cellmask_{i}.png", cell_mask)

### Get HPAv24 Images URL Images

In [6]:
df = pd.read_csv("v24_hpa_subcellular_location_expanded.csv")
df = df[['Gene ID', 'Cell Line', 'Location', 'Image']]
df['Image'] = df['Image'].apply(lambda x: x.split('_blue_red_green.jpg')[0])

print(len(df), "rows of HPA data fetched.")
df.head(10)

83360 rows of HPA data fetched.


,Gene ID,Cell Line,Location,Image
0,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_2
1,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_4
2,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_1...
3,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_3...
4,ENSG00000000003,U2OS,['cytosol (GO:0005829)'],https://images.proteinatlas.org/4109/23_H11_1
5,ENSG00000000003,U2OS,['cytosol (GO:0005829)'],https://images.proteinatlas.org/4109/23_H11_2
6,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_1
7,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_2
8,ENSG00000000457,SK-MEL-30,[],https://images.proteinatlas.org/72383/2142_F7_1
9,ENSG00000000457,SK-MEL-30,[],https://images.proteinatlas.org/72383/2142_F7_2


In [7]:
# Remove Kaggle train and test set images
kaggle_train_ids = pd.read_csv('/home/jumidlej/git-projects/hpa-single-cell-classification/hpa-datasets/train_kaggle_2021/train.csv')
kaggle_train_ids = set(kaggle_train_ids['ID'].values)
print(len(kaggle_train_ids), "unique image IDs in Kaggle train set.")
print(list(kaggle_train_ids)[:5])
kaggle_test_ids = pd.read_csv('/home/jumidlej/git-projects/hpa-single-cell-classification/hpa-datasets/train_kaggle_2021/sample_submission.csv')
kaggle_test_ids = set(kaggle_test_ids['ID'].values)
print(len(kaggle_test_ids), "unique image IDs in Kaggle test set.")
print(list(kaggle_test_ids)[:5])

# Create ID label on df
df['ID'] = df['Image'].apply(lambda x: x.split('/')[-1])

# Remove Kaggle images from df
df = df[~df['ID'].isin(kaggle_train_ids)]
df = df[~df['ID'].isin(kaggle_test_ids)]

print(len(df), "rows of HPA data after removing Kaggle images.")
df.head(10)

21806 unique image IDs in Kaggle train set.
['303dc0ac-bbbb-11e8-b2ba-ac1f6b6435d0', 'e0dde7f2-bbac-11e8-b2ba-ac1f6b6435d0', '59860238-bbb8-11e8-b2ba-ac1f6b6435d0', 'c4908034-bb9b-11e8-b2b9-ac1f6b6435d0', 'f38ad554-bba7-11e8-b2ba-ac1f6b6435d0']
559 unique image IDs in Kaggle test set.
['b95ba845-7a57-4cf2-9f89-171e04b44518', 'e5858744-e8cd-4628-b7fd-61d2a624172b', '92cccd0a-8a4a-49bd-9b71-149ff15057b7', '6367c9fd-073b-43c4-afe9-6dcd4ebba1da', '8b6e2195-8658-4262-aa21-4bb769fe2b9f']
83360 rows of HPA data after removing Kaggle images.


,Gene ID,Cell Line,Location,Image,ID
0,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_2,1832_C1_2
1,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_4,1832_C1_4
2,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_1...,1843_B2_17_cr5af971a263864
3,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_3...,1843_B2_30_cr5af971a2648d6
4,ENSG00000000003,U2OS,['cytosol (GO:0005829)'],https://images.proteinatlas.org/4109/23_H11_1,23_H11_1
5,ENSG00000000003,U2OS,['cytosol (GO:0005829)'],https://images.proteinatlas.org/4109/23_H11_2,23_H11_2
6,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_1,2038_G11_1
7,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_2,2038_G11_2
8,ENSG00000000457,SK-MEL-30,[],https://images.proteinatlas.org/72383/2142_F7_1,2142_F7_1
9,ENSG00000000457,SK-MEL-30,[],https://images.proteinatlas.org/72383/2142_F7_2,2142_F7_2


In [8]:
# Remove images that contains only "['cytosol (GO:0005829)']"" or "['nucleoplasm (GO:0005654)']"
df = df[~df['Location'].isin([
    "['cytosol (GO:0005829)']", 
    "['nucleoplasm (GO:0005654)']", 
    "['cytosol (GO:0005829)', 'nucleoplasm (GO:0005654)']",
    "['nucleoplasm (GO:0005654)', 'cytosol (GO:0005829)']"
])]
print(len(df), "rows of HPA data after removing cytosol and nucleoplasm only images.")
df.head(10)

57698 rows of HPA data after removing cytosol and nucleoplasm only images.


,Gene ID,Cell Line,Location,Image,ID
0,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_2,1832_C1_2
1,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_4,1832_C1_4
2,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_1...,1843_B2_17_cr5af971a263864
3,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_3...,1843_B2_30_cr5af971a2648d6
6,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_1,2038_G11_1
7,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_2,2038_G11_2
8,ENSG00000000457,SK-MEL-30,[],https://images.proteinatlas.org/72383/2142_F7_1,2142_F7_1
9,ENSG00000000457,SK-MEL-30,[],https://images.proteinatlas.org/72383/2142_F7_2,2142_F7_2
10,ENSG00000000457,SuSa,"['endoplasmic reticulum (GO:0005783)', 'golgi ...",https://images.proteinatlas.org/72383/2091_H8_1,2091_H8_1
11,ENSG00000000457,SuSa,"['endoplasmic reticulum (GO:0005783)', 'golgi ...",https://images.proteinatlas.org/72383/2091_H8_2,2091_H8_2


### Download a Image

In [22]:
import tifffile

def read_tif_from_url(url):
    success = False
    try:
        r = requests.get(url)
        r.raise_for_status()
        with gzip.open(io.BytesIO(r.content)) as f:
            buf = f.read()
    except Exception as e:
        print(f"⚠️ Error downloading or decompressing {url}: {e}")
        return success, url

    try:
        img = tifffile.imread(io.BytesIO(buf), is_ome=False)
    except Exception as e:
        print(f"⚠️ Error with OME metadata in {url}: {e}")
        with tifffile.TiffFile(io.BytesIO(buf)) as tif:
            try:
                img = tif.pages[0].asarray()
            except Exception as e:
                print(f"⚠️ Error reading TIFF page in {url}: {e}")
                return success, url
    
    success = True
    return success, img

In [23]:
def get_hpa_image(base_img_url):
    """
    Downloads the 4 color channels for an HPA image, converts them to uint8,
    and stacks them into a single multi-channel NumPy array.

    Args:
        base_img_url (str): The base URL/name for the image, without the color suffix.
                           e.g., "https://images.proteinatlas.org/4109/1832_C1_4"

    Returns:
        A NumPy array of shape (Height, Width, Channels), or None if all channels fail.
    """
    colors = ['red', 'green', 'blue', 'yellow']
    
    # 1. Collect successfully downloaded channels in a list
    channels = []
    for color in colors:
        img_url = f'{base_img_url}_{color}.tif.gz'
        success, img_color = read_tif_from_url(img_url)
        if success:
            if img_color.dtype == np.uint16:
                img_color = (img_color / 256).astype(np.uint8)
            channels.append(img_color)
        else:
            return None

    if not channels:
        return None
    else:
        hpa_img = np.stack(channels, axis=-1)
        return hpa_img

In [11]:
for i, row in df[1:2].iterrows():
    base_img_url = row.Image
    hpa_img = get_hpa_image(base_img_url)
    if hpa_img is not None:
        print(f"Image shape: {hpa_img.shape}, dtype: {hpa_img.dtype}")

Image shape: (2048, 2048, 4), dtype: uint8


### Full Pipeline

In [12]:
# -------------------------------
# CONFIG
# -------------------------------
IMAGE_SIZE = 2048
SAVE_DIR = "/home/jumidlej/git-projects/hpa-single-cell-classification/hpa-datasets/v24/images"
CELL_CROP_DIR = os.path.join(SAVE_DIR, "cell_crops")
FULL_IMG_DIR = os.path.join(SAVE_DIR, "full")

os.makedirs(CELL_CROP_DIR, exist_ok=True)
os.makedirs(FULL_IMG_DIR, exist_ok=True)

# HPA pretrained models (download them first or update the paths)
NUC_MODEL = "./nuclei-model.pth"
CELL_MODEL = "./cell-model.pth"

# Colors in HPA images
COLORS = ["red", "green", "blue", "yellow"]

# -------------------------------
# HELPER FUNCTIONS
# -------------------------------
def squarify(M,val):
    (a,b,c)=M.shape
    if a>b:
        padding=((0,0),((a-b)//2,a-b-(a-b)//2),(0, 0))
    else:
        padding=(((b-a)//2,b-a-(b-a)//2),(0,0),(0, 0))
    return np.pad(M,padding,mode='constant',constant_values=val)


In [25]:
df

,Gene ID,Cell Line,Location,Image,ID
0,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_2,1832_C1_2
1,ENSG00000000003,CACO-2,"['nucleoli fibrillar center (GO:0001650)', 'cy...",https://images.proteinatlas.org/4109/1832_C1_4,1832_C1_4
2,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_1...,1843_B2_17_cr5af971a263864
3,ENSG00000000003,RT-4,"['nucleoli fibrillar center (GO:0001650)', 'ce...",https://images.proteinatlas.org/4109/1843_B2_3...,1843_B2_30_cr5af971a2648d6
6,ENSG00000000457,OE19,"['golgi apparatus (GO:0005794)', 'cytosol (GO:...",https://images.proteinatlas.org/72383/2038_G11_1,2038_G11_1
...,...,...,...,...,...
83353,ENSG00000291237,U2OS,['mitochondria (GO:0005739)'],https://images.proteinatlas.org/1814/1865_C5_2,1865_C5_2
83354,ENSG00000291316,HeLa,[],https://images.proteinatlas.org/59543/1282_B2_2,1282_B2_2
83355,ENSG00000291316,HeLa,[],https://images.proteinatlas.org/59543/1282_B2_4,1282_B2_4
83358,ENSG00000291316,U2OS,[],https://images.proteinatlas.org/59543/1018_C5_1,1018_C5_1


In [ ]:
def plot_rgby_channels(img, title="HPA Location Image"):
    """
    img: numpy array of shape (H, W, 4) with channels in RGBY order
    """

    colors = {
        "Red":   [1, 0, 0],   # RGB for red
        "Green": [0, 1, 0],   # RGB for green
        "Blue":  [0, 0, 1],   # RGB for blue
        "Yellow":[1, 1, 0]    # RGB for yellow
    }

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(title, fontsize=16)

    for i, (name, color) in enumerate(colors.items()):
        channel = img[:, :, i]
        
        # Normalize channel to [0,1] for display
        channel = channel.astype(float)
        if channel.max() > 0:
            channel /= channel.max()
        
        # Create RGB image tinted with the right color
        rgb = np.zeros((*channel.shape, 3))
        for c in range(3):
            rgb[..., c] = channel * color[c]

        axes[i].imshow(rgb)
        axes[i].set_title(name)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
img = imread('/home/jumidlej/git-projects/hpa-single-cell-classification/hpa-datasets/v24/images/full/1832_C1_4.png')
plot_rgby_channels(img)

In [ ]:
img = imread('/home/jumidlej/git-projects/hpa-single-cell-classification/hpa-datasets/v24/images/cell_crops/1832_C1_4_1.png')
plot_rgby_channels(img)

In [18]:
df[df['ID']=='2238_C6_36']

,Gene ID,Cell Line,Location,Image,ID
99,ENSG00000002549,Sperm,"['calyx (GO:0120238)', 'mid piece (GO:0097225)...",https://images.proteinatlas.org/29606/2238_C6_36,2238_C6_36


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

failed_images = []
def process_image_row(row):
    # Check if the image already exists on full image folder
    iid = row.Image.split('/')[-1]
    if os.path.exists(f'{FULL_IMG_DIR}/{iid}.png'):
        print('Image already exists:', iid)
        return

    base_img_url = row.Image
    iid = base_img_url.split('/')[-1]
    hpa_img = get_hpa_image(base_img_url)
    if hpa_img is None:
        print('failed', base_img_url)
        failed_images.append(base_img_url)
        return f"Failed to download {base_img_url}"
    else:
        # Image resizing
        if not hpa_img.shape[0] == IMAGE_SIZE:
            hpa_img = cv2.resize(hpa_img, (IMAGE_SIZE, IMAGE_SIZE))

        # Segment cell and get cells masks
        nuclei_seg = segmentator.pred_nuclei([hpa_img[..., 2]])  # blue channel for nuclei
        cell_seg = segmentator.pred_cells([hpa_img[..., [0, 3, 2]]], precombined=True) # order: microtubules, ER, nuclei R Y B
        nuclei_mask, cell_mask = label_cell(nuclei_seg[0], cell_seg[0])

        mask = cell_mask
        for i in range(1, mask.max()):
            sub_mask = cv2.resize((mask == i).astype(np.uint8), (IMAGE_SIZE, IMAGE_SIZE))
            x, y = np.where(sub_mask == 1)
            sub = hpa_img[x.min(): x.max(), y.min(): y.max()]
            crop_sub_mask = sub_mask[x.min(): x.max(), y.min(): y.max()]
            crop_sub_mask = np.repeat(crop_sub_mask[:, :, np.newaxis], 4, axis=2)
            r = sub * crop_sub_mask
            r = squarify(r, 0)
            if r.shape[0] > 256:
                r = cv2.resize(r, (256, 256))
            # Save on Cell Crop folder
            imsave(f'{CELL_CROP_DIR}/{iid}_{i}.png', r)

        # Rezise to 512 and save on Full Image folder
        if hpa_img.shape[0] > 512:
            hpa_img = cv2.resize(hpa_img, (512, 512))
        imsave(f'{FULL_IMG_DIR}/{iid}.png', hpa_img)


# 2. Use the ThreadPoolExecutor to run the function in parallel
# Set max_workers to the number of parallel downloads you want (e.g., 8, 16)
MAX_WORKERS = 1

#  image_rows = [row for row in df.itertuples(index=False)]
image_rows = list(df[df['ID']=='2238_C6_36'].itertuples(index=False))
print(f"Processing {len(image_rows)} images with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Use tqdm to track progress
    results = list(tqdm(executor.map(process_image_row, image_rows), total=len(image_rows)))

# Optionally, print statuses
for r in results:
    if "Failed" in r:
        print(r)

Processing 1 images with 1 workers...


100%|██████████| 1/1 [00:01<00:00,  1.52s/it]

⚠️ Error downloading or decompressing https://images.proteinatlas.org/29606/2238_C6_36_red.tif.gz: 400 Client Error: Bad Request for url: https://www.proteinatlas.org/download_file.php?filename=/29606/2238_C6_36_red&format=tif.gz
failed https://images.proteinatlas.org/29606/2238_C6_36
Failed to download https://images.proteinatlas.org/29606/2238_C6_36
